In [1]:
import os 

In [2]:
os.chdir("..\.")

In [3]:
import torch
from torch import nn
import torch.nn.functional as F

In [36]:
torch.manual_seed(1337)
with open ("data\gpt_train.txt","r") as file:
    text    = file.read()


# All the unique characters that occur in this text 
chars       = sorted((set(text)))
vocab_size  = len(chars)

## creating mapping from characters to integers 
stoi    = {ch:i for i,ch in enumerate(chars)}
itos    = dict(enumerate(chars))
encode  = lambda word: [stoi[i] for i in word]
decode  = lambda integers: "".join(itos[int(i)] for i in integers)



## Let's encode entire dataset of file. 
data = torch.tensor(encode(text),dtype=torch.long,device="cuda")

## Lets split the data into train and val dataset 
n   = int(0.9 * len(data))
train_data  = data[:n]
val_data    = data[n:]

## Multi-Head Attention

Its about applyig multiple attention  layer in parallel, and concatenate their result in parallel. 

In [22]:
import torch
from tqdm import tqdm
from torch import nn
from torch.nn import functional as F
from common.data_processing import get_batch     


batch_size  = 4     # B 
block_size  = 8     # T
n_emd       = 32    # C 
device      = "cuda" if torch.cuda.is_available() else "cpu"
eval_interval   = 300
learning_rate   = 1e-3 
max_iters       = 5000 
eval_iter       = 200

vocab_size      = 65 

In [5]:
class Head(nn.Module):
    def __init__(self,head_size):
        super().__init__()
        self.key    = nn.Linear(n_emd,head_size,bias=False)   # key projection
        self.query  = nn.Linear(n_emd,head_size,bias=False)   # query projection
        self.value  = nn.Linear(n_emd,head_size,bias=False)   # value projection 
        self.register_buffer('tril',torch.tril(torch.ones(block_size,block_size)))
    def forward(self,x):
        _,T,C   = x.shape
        k       = self.key(x)   # (B,T,head_size) ==> 4,8,16
        q       = self.query(x) # (B,T,head_size) ==> 4,8,16   
        ## compute attention score(affinities)
        wei     = q @ k.transpose(-2,-1) * C ** -0.5   # (B,T,c) @ (B,C,T) ==> (B,T,T)
        wei     = wei.masked_fill(self.tril[:T,:T] == 0, float('-inf'))
        wei     = F.softmax(wei,dim=-1)
        # perform the weighted aggregation of the value
        v       = self.value(x) # (B,T,head_size) ==> 4,8,16
        out     = wei @ v       # (B,T,T) @ (B,T,head_size) ==> (B,T,head_size) 
        return out 

In [ ]:
class MultiHeadAttention(nn.Module):
    def __init__(self, num_heads,head_size):
        super().__init__()
        self.heads   = nn.ModuleList([Head(head_size) for _ in range(num_heads)])
    def forward(self,x):
        return torch.cat([head(x) for head in self.heads],dim=-1)   # concatenate in the channel dimension 

In [25]:
class BiGramLanguageModel(nn.Module):

    def __init__(self):
        super().__init__()
        # each token directly reads off the logits for next token from a lookup table 
        self.token_embedding_table      = nn.Embedding(vocab_size,n_emd)
        self.position_embedding_table   = nn.Embedding(block_size,n_emd)
        self.self_attention_head        = MultiHeadAttention(num_heads=4,head_size=n_emd//4)    # 1) head_size = n_emd / num_heads  
        self.lm_head                    = nn.Linear(n_emd,vocab_size)       
        

    def forward(self,idx,target=None):
        B,T         = idx.shape 
        # index and target are both (B,T) tensor of integers
        tok_emb     = self.token_embedding_table(idx)   # its arrange in the shape of ==================================================================>> (B,T,C)
        pos_emb     = self.position_embedding_table(torch.arange(T,device=device)) # This embdding gives the idea of where word is belong in a sentence=>> (T,C) 
        x           = tok_emb + pos_emb                 # combined representation of token and its position.======Broadcasting apply=>> (B,T,C) + (T,C) == (B,T,C) 
        x           = self.self_attention_head(x)       # (B,T,C)
        logits      = self.lm_head(x)                   # for getting token_emb to logits we need linear layer,shape ===================================>> (B,T,vocab_size) 

        if target is None:
            loss    = None
        else:    
            B,T,C   = logits.shape
            logits  = logits.view(B*T,C)    # cross entropy input expectation is (minibatch,C)
            target  = target.view(B*T)      
            loss    = nn.functional.cross_entropy(logits,target)
        return logits,loss
    
    def generate(self,idx,max_new_tokens):
        # idx is (B,T) array of indices in the current context. 
        for _ in range(max_new_tokens):
            idx_cond    = idx[:,-block_size:]                   # crop idx to last block_size tokens
            logits,loss = self.forward(idx_cond)                # Get the prediction ==>  (B,T,C)
            logits      = logits[:,-1,:]                        # focus only on last time step  ==>  (B,C)
            probs       = nn.functional.softmax(logits,dim=-1)  # (B,C)
            # sample from the distribution 
            idx_next    = torch.multinomial(probs,num_samples=1)    # (B,1)
            # append sample index to running sequence 
            idx         = torch.cat([idx,idx_next],dim=1)           # (B,T+1)
        return idx


In [33]:
x = torch.randn(4,8,16)
x.shape

torch.Size([4, 8, 16])

In [34]:
model = BiGramLanguageModel()
model = model.to("cuda:0")

In [35]:

@torch.no_grad()
def estimate_loss():
    out = {}
    model.eval()
    for split in ["train","val"]:
        losses = torch.zeros(eval_iter)
        for k in range(eval_iter):
            X,Y         = get_batch("train")
            logits,loss = model(X,Y)
            losses[k]   = loss.item()   
        out[split] = losses.mean()
    model.train()
    return out

In [37]:

optimizer   = torch.optim.AdamW(model.parameters(),lr=learning_rate)


for iter in range(max_iters):
    if iter % eval_interval == 0:
        losses  = estimate_loss()
        print(f"step {iter}: train loss {losses['train']:.4f},val loss {losses['val']:.4f}")

    #sample a batch of data
    xb,yb = get_batch("train")
    #evaluate the loss
    logits,loss = model(xb,yb)
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()


context = torch.zeros((1,1),dtype=torch.long,device="cuda:0")
print(decode(model.generate(context,max_new_tokens=500)[0]))

c:\Users\Asus\anaconda3\envs\llm_env\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


step 0: train loss 4.2240,val loss 4.2294
step 300: train loss 3.0590,val loss 3.1049
step 600: train loss 2.8883,val loss 2.8633
step 900: train loss 2.7043,val loss 2.7295
step 1200: train loss 2.6279,val loss 2.6677
step 1500: train loss 2.5762,val loss 2.5724
step 1800: train loss 2.5099,val loss 2.5296
step 2100: train loss 2.5264,val loss 2.5030
step 2400: train loss 2.5077,val loss 2.5000
step 2700: train loss 2.4888,val loss 2.4838
step 3000: train loss 2.4510,val loss 2.4521
step 3300: train loss 2.4404,val loss 2.4401
step 3600: train loss 2.4294,val loss 2.4197
step 3900: train loss 2.4289,val loss 2.4332
step 4200: train loss 2.4050,val loss 2.4158
step 4500: train loss 2.4007,val loss 2.3773
step 4800: train loss 2.3943,val loss 2.3821

Whent ak bridcowe,
This a, be mad sen bobe doe.
S:
O-3 my dalilauss ar bthie usqhe he.
War delas ate awice my.

HDEESAn zoto he owth, tof in he ce mil; dill, aes iree sen cin lat Hot drov te, and Win ner og yombs!
 lolind me lit-hulr cecher